# Capstone — Prioritizing Pages for Editorial Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MTahaR-dev/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** We are building this capstone step by step. The current completed step prepares five features and a separate historical outcome target. No model has been trained in this notebook yet.

Run the code cells in order. Python 3, pandas, NumPy, and DuckDB are needed. The March warehouse file is reused locally; the setup also supports a gated download with a securely configured Hugging Face token, as in Week 3. Generated data caches stay out of Git. The notebook and aggregate JSON receipt are safe to commit.

## 1. Question

**Can past search-performance signals rank pages by later impression-decline risk better than a transparent rule, helping an editor choose which pages to review first?**

An editor has limited review capacity. The output will be a ranked page list with measured reasons, supporting investigation and a possible refresh. A wrong recommendation can waste review time or encourage unnecessary changes; a missed candidate may leave a problem unattended. The outcome measures decline, not whether editing would help. Observed associations and decision-support recommendations are the intended claims.

## 2. Data — build the five-feature table

We use `fact_content_daily_performance`, March 2026, from the FlyRank warehouse release `v20260703`, pinned repository revision `50cbf7c3909d07be4d1b5906b4d09e882e5acbf2`.

**Raw row:** one report date × client × content item. **Prepared row:** one client/content item at the March 18 decision point.

| Dates | Role |
|---|---|
| March 1–7 | First week of past measurements |
| March 8–14 | Second week of past measurements |
| March 15–17 | Reporting buffer; excluded |
| March 18–31 | Later outcome evidence; target only |

Keep a past day only when `gsc_data_available IS TRUE` and impressions/clicks are observed and non-negative. Require all 14 past days and at least 100 past impressions. That defines the past-only scoring population. A page needs all 14 usable later days to enter the labeled training/evaluation population; future coverage is not a scoring input.

Only positive-impression days with a positive, observed position contribute to average position. Unavailable position remains missing; any imputation will be fitted on training clients in a later step. No GA4, AI-referral, product-score, text, or incompatible query-table fields enter these five features. June remains untouched. The three-day buffer is an explicit assumption because historical ingestion timestamps are unavailable.

In [1]:
# Install only missing libraries; no credentials appear in this command.
import importlib.util
import subprocess
import sys

packages = {"duckdb": "duckdb", "pandas": "pandas"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

MONTH = "2026-03"
REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
FILENAME = f"fact_content_daily_performance/month={MONTH}/data_0.parquet"

# Find a repo root if present; otherwise use the current Colab folder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/ml-intern-dataset-and-lane-guide.md").is_file()), Path.cwd())
local_candidates = [
    root / "data/warehouse" / f"month={MONTH}" / "data_0.parquet",
    root / "data/warehouse" / FILENAME,
]
month_file = next((p for p in local_candidates if p.is_file()), None)

if month_file is None:
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import get_token, hf_hub_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    token = token or get_token()
    if not token:
        raise RuntimeError("Warehouse access is required: set HF_TOKEN securely, "
                           "use a local Hugging Face login, or supply the March parquet file. "
                           "Do not paste a token into this notebook.")
    month_file = Path(hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=FILENAME,
        revision=REVISION, token=token, cache_dir=str(root / "data/warehouse_cache"),
    ))
    del token

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")
temp_folder = root / "data/warehouse_cache/duckdb_tmp"
temp_folder.mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = ?", [str(temp_folder)])
# This is a local cached file, so repeat verification does not rescan Hugging Face.
con.read_parquet(str(month_file)).create_view("march_raw")
required_columns = {
    "report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
    "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position",
}
observed_columns = set(con.table("march_raw").columns)
assert required_columns <= observed_columns, "Warehouse schema differs; inspect before proceeding."
print("Loaded the cached March partition; required field names verified.")

Loaded the cached March partition; required field names verified.


### A. Combine daily records into one past-window row per page

The query below looks only at March 1–14. It sums impressions and clicks, calculates the two weekly impression totals, and collects valid position measurements. It never reads the later outcome window.

In [2]:
past_query = """
SELECT client_hash_id, content_hash_id, COUNT(*) AS feature_days,
       MIN(report_date) AS feature_start, MAX(report_date) AS feature_end,
       SUM(gsc_impressions) AS feature_impressions,
       SUM(gsc_clicks) AS feature_clicks,
       SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07') AS first_week_impressions,
       SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-08') AS second_week_impressions,
       SUM(gsc_avg_position * gsc_impressions) FILTER (
           WHERE gsc_impressions > 0 AND gsc_avg_position > 0) AS position_numerator,
       SUM(gsc_impressions) FILTER (
           WHERE gsc_impressions > 0 AND gsc_avg_position > 0) AS position_denominator
FROM march_raw
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
  AND gsc_impressions >= 0 AND gsc_clicks >= 0
GROUP BY client_hash_id, content_hash_id
HAVING feature_days = 14 AND COUNT(DISTINCT report_date) = 14 AND feature_impressions >= 100
ORDER BY client_hash_id, content_hash_id
"""
past_pages = con.execute(past_query).df()
past_pages["decision_date"] = pd.Timestamp("2026-03-18")
assert not past_pages.duplicated(["client_hash_id", "content_hash_id", "decision_date"]).any()
assert past_pages.feature_days.eq(14).all()
assert pd.to_datetime(past_pages.feature_start).eq(pd.Timestamp("2026-03-01")).all()
assert pd.to_datetime(past_pages.feature_end).eq(pd.Timestamp("2026-03-14")).all()
assert past_pages.feature_impressions.ge(100).all()
print(f"Prepared {len(past_pages):,} past-only page rows across "
      f"{past_pages.client_hash_id.nunique()} clients.")
past_pages[["feature_impressions", "feature_clicks",
            "first_week_impressions", "second_week_impressions"]].head(5)

Prepared 60,534 past-only page rows across 33 clients.


,feature_impressions,feature_clicks,first_week_impressions,second_week_impressions
0,114.0,1.0,52.0,62.0
1,272.0,0.0,138.0,134.0
2,399.0,2.0,214.0,185.0
3,197.0,0.0,114.0,83.0
4,169.0,0.0,82.0,87.0


### B. Calculate the five inputs

`X_past` is a dataframe: rows are pages, columns are the five numbers a model could receive. IDs and future outcomes stay outside it.

| Feature | Calculation | Meaning |
|---|---|---|
| `log_impressions` | `log(1 + past impressions)` | Past search exposure, on a compressed numeric scale |
| `log_clicks` | `log(1 + past clicks)` | Click volume, on the same compressed scale |
| `ctr_pct` | `100 × past clicks / past impressions` | Percentage of exposure that became clicks |
| `weighted_position` | Impression-weighted average of valid past daily positions | Search-position context; lower is better |
| `prior_week_change_pct` | `100 × (week 2 − week 1) / week 1` | Whether visibility was already falling or rising |

Every calculation uses March 1–14 only. A zero first-week denominator becomes missing instead of an infinite percentage. Position is an aggregate approximation from daily averages; exact within-day weighting cannot be recovered. No imputation, scaling fit, or model fitting happens in this step.

In [3]:
FEATURE_COLUMNS = [
    "log_impressions", "log_clicks", "ctr_pct",
    "weighted_position", "prior_week_change_pct",
]
X_past = pd.DataFrame({
    "log_impressions": np.log1p(past_pages.feature_impressions),
    "log_clicks": np.log1p(past_pages.feature_clicks),
    "ctr_pct": 100 * past_pages.feature_clicks / past_pages.feature_impressions,
    "weighted_position": past_pages.position_numerator / past_pages.position_denominator,
    "prior_week_change_pct": 100 * (
        past_pages.second_week_impressions - past_pages.first_week_impressions
    ) / past_pages.first_week_impressions.replace(0, np.nan),
}).replace([np.inf, -np.inf], np.nan)
assert list(X_past.columns) == FEATURE_COLUMNS and X_past.shape[1] == 5
print(f"Past-only feature table: {X_past.shape[0]:,} rows × {X_past.shape[1]} features.")
X_past.head(5).round(4)

Past-only feature table: 60,534 rows × 5 features.


,log_impressions,log_clicks,ctr_pct,weighted_position,prior_week_change_pct
0,4.7449,0.6931,0.8772,12.4211,19.2308
1,5.6095,0.0000,0.0000,12.4669,-2.8986
2,5.9915,1.0986,0.5013,11.7068,-13.5514
3,5.2883,0.0000,0.0000,8.9188,-27.1930
4,5.1358,0.0000,0.0000,18.5562,6.0976


### C. Build the answer column separately

`y` is the answer the model will learn from: **1** if impressions in March 18–31 fell by more than 20% relative to March 1–14; **0** otherwise. These are equal-length 14-day windows. The later counts never become input features.

Not all pages have a complete later window. We retain all eligible past pages in `X_past` for scoring, while `X` and `y` contain only pages with a measurable complete outcome. This is a retrospective complete-case subset and can bias evaluation toward better-tracked pages. `page_context` stores identifiers and the decision date separately for grouping and recommendations.

In [4]:
outcome_query = """
SELECT client_hash_id, content_hash_id, COUNT(*) AS outcome_days,
       MIN(report_date) AS outcome_start, MAX(report_date) AS outcome_end,
       SUM(gsc_impressions) AS outcome_impressions
FROM march_raw
WHERE report_date BETWEEN DATE '2026-03-18' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
  AND gsc_impressions >= 0 AND gsc_clicks >= 0
GROUP BY client_hash_id, content_hash_id
HAVING outcome_days = 14 AND COUNT(DISTINCT report_date) = 14
"""
later_outcomes = con.execute(outcome_query).df()
page_data = past_pages.merge(
    later_outcomes, on=["client_hash_id", "content_hash_id"],
    how="left", validate="one_to_one", sort=False,
)
# Prove that merging target evidence did not reorder the past feature rows.
assert page_data.content_hash_id.tolist() == past_pages.content_hash_id.tolist()
assert page_data.client_hash_id.tolist() == past_pages.client_hash_id.tolist()
has_target = page_data.outcome_days.eq(14)
labeled_pages = page_data.loc[has_target].reset_index(drop=True)
X = X_past.loc[has_target].reset_index(drop=True).copy()
y = (
    labeled_pages.outcome_impressions.lt(0.8 * labeled_pages.feature_impressions)
    .astype("int64").rename("future_decline_proxy")
)
page_context = labeled_pages[
    ["client_hash_id", "content_hash_id", "decision_date"]
].copy()
groups = page_context.client_hash_id.copy()
assert len(X) == len(y) == len(page_context)
assert X.index.equals(y.index) and X.index.equals(page_context.index)
assert pd.to_datetime(labeled_pages.outcome_start).eq(pd.Timestamp("2026-03-18")).all()
assert pd.to_datetime(labeled_pages.outcome_end).eq(pd.Timestamp("2026-03-31")).all()
assert set(y.unique()) == {0, 1}
assert list(X.columns) == FEATURE_COLUMNS
assert not any(name in X.columns for name in ["client_hash_id", "content_hash_id",
               "future_decline_proxy", "outcome_impressions", "outcome_days"])
print(f"X: {len(X):,} rows × five features; y: {len(y):,} answers.")
print(f"Measured later decline: {int(y.sum()):,} pages ({y.mean():.2%}); "
      f"no such decline: {int(y.eq(0).sum()):,} pages.")
print(f"Clients: {groups.nunique()}; past-only pages without a complete target: "
      f"{int((~has_target).sum()):,}.")
X.head(5).round(4)

X: 57,459 rows × five features; y: 57,459 answers.
Measured later decline: 21,053 pages (36.64%); no such decline: 36,406 pages.
Clients: 32; past-only pages without a complete target: 3,075.


,log_impressions,log_clicks,ctr_pct,weighted_position,prior_week_change_pct
0,4.7449,0.6931,0.8772,12.4211,19.2308
1,5.6095,0.0000,0.0000,12.4669,-2.8986
2,5.9915,1.0986,0.5013,11.7068,-13.5514
3,7.3645,2.7081,0.8872,7.3086,-63.3766
4,7.1678,0.0000,0.0000,56.0039,-15.6472


### D. Read one prepared row

The next small table shows original counts beside selected calculated values and the separate answer. It is a learning preview, **not** the model input table: the model receives only `X`'s five columns. The row number matches `X`, `y`, and `page_context`.

For example, CTR is a percentage: one click from 114 impressions is about 0.877%, not 87.7%. A positive past weekly change can still be followed by a later decline; a feature is a possible clue, not a guaranteed answer.

In [5]:
learning_preview = pd.DataFrame({
    "past_impressions": labeled_pages.feature_impressions,
    "past_clicks": labeled_pages.feature_clicks,
    "week1_impressions": labeled_pages.first_week_impressions,
    "week2_impressions": labeled_pages.second_week_impressions,
    "ctr_pct": X.ctr_pct,
    "prior_week_change_pct": X.prior_week_change_pct,
    "later_impressions_TARGET_ONLY": labeled_pages.outcome_impressions,
    "y_TARGET_ONLY": y,
})
learning_preview.head(5).round(4)

,past_impressions,past_clicks,week1_impressions,week2_impressions,ctr_pct,prior_week_change_pct,later_impressions_TARGET_ONLY,y_TARGET_ONLY
0,114.0,1.0,52.0,62.0,0.8772,19.2308,189.0,0
1,272.0,0.0,138.0,134.0,0.0000,-2.8986,158.0,1
2,399.0,2.0,214.0,185.0,0.5013,-13.5514,233.0,1
3,1578.0,14.0,1155.0,423.0,0.8872,-63.3766,1801.0,0
4,1296.0,0.0,703.0,593.0,0.0000,-15.6472,601.0,1


### E. Check and save this preparation step

We check missing values, variation, alignment, and consistency with the existing Week 4 population. The three CSV files are local caches in the repo's ignored work area; they must not be committed. The JSON receipt contains aggregate information, making it suitable to commit with the notebook.

The data step prepares inputs; splitting clients, fitting any preprocessing, training a model, and comparing it with the frozen rule are the next steps.

In [6]:
import json
import hashlib

quality = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "missing_rows": [int(X[c].isna().sum()) for c in FEATURE_COLUMNS],
    "distinct_values": [int(X[c].nunique()) for c in FEATURE_COLUMNS],
})
assert quality.distinct_values.gt(1).all(), "A feature is constant; investigate before training."
assert not np.isinf(X.to_numpy(dtype=float)).any()

output_dir = root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
X.to_csv(output_dir / "capstone_features.csv", index=False)
y.to_frame().to_csv(output_dir / "capstone_target.csv", index=False)
page_context.to_csv(output_dir / "capstone_page_context.csv", index=False)

baseline_path = output_dir / "w04_baseline_metrics.json"
if baseline_path.is_file():
    baseline_receipt = json.loads(baseline_path.read_text(encoding="utf-8"))
    assert baseline_receipt["labeled_pages"] == len(X)
    assert baseline_receipt["past_only_scoring_pages"] == len(X_past)

identity_rows = pd.concat([page_context, y], axis=1).to_csv(index=False)
feature_receipt = {
    "step": "five-feature table; model training pending",
    "warehouse_revision": REVISION, "month": MONTH,
    "decision_date": "2026-03-18",
    "feature_window": ["2026-03-01", "2026-03-14"],
    "outcome_window": ["2026-03-18", "2026-03-31"],
    "minimum_feature_impressions": 100, "required_days_per_window": 14,
    "feature_names": FEATURE_COLUMNS,
    "past_only_pages": len(X_past), "labeled_pages": len(X),
    "clients": int(groups.nunique()), "positive_targets": int(y.sum()),
    "negative_targets": int(y.eq(0).sum()),
    "pages_without_complete_target": int((~has_target).sum()),
    "quality": quality.to_dict(orient="records"),
    "identity_and_target_sha256": hashlib.sha256(identity_rows.encode("utf-8")).hexdigest(),
    "notes": "Targets and IDs are separate from X. Missing position stays missing until train-only preprocessing. March complete-case slice; historical ingestion times unknown.",
}
(output_dir / "capstone_feature_table.json").write_text(
    json.dumps(feature_receipt, indent=2, allow_nan=False) + chr(10), encoding="utf-8"
)
print("Saved three local CSV caches and the aggregate capstone_feature_table.json receipt.")
print("All five features have variation; target and context alignment verified.")
quality

Saved three local CSV caches and the aggregate capstone_feature_table.json receipt.
All five features have variation; target and context alignment verified.


,feature,missing_rows,distinct_values
0,log_impressions,0,8318
1,log_clicks,0,223
2,ctr_pct,0,16224
3,weighted_position,0,55457
4,prior_week_change_pct,0,39897


## 3. Methodology

**Feature preparation completed.** The five inputs, windows, target, availability rules, and exclusions are defined above. Next we will split whole clients, fit any preprocessing on training data only, choose a simple model, and compare its ranking with the frozen Week 4 baseline on the same evaluation population. No model training has occurred in this notebook yet.

## 4. Results (vs baseline)

To be completed after model training and evaluation.

## 5. Limitations

Current data limitations: one March decision window, unequal client inventories, retrospective complete-case labels, and unknown historical ingestion times. These measurements cannot identify the cause of decline or prove the benefit of a refresh. We will expand this section after validation.

## 6. Ranked recommendations

To be completed after the model/rule comparison and skeptical review.

## 7. Artifacts the paper embeds

To be completed after results: figures, tables, and reproducibility links.

## Progress and final self-check

- [x] Research question and decision are stated.
- [x] Five features are prepared from past measurements only.
- [x] The target and identifiers are separate from model inputs.
- [x] Data preparation cells executed successfully and their outputs are retained.
- [ ] Client split and train-only preprocessing are complete.
- [ ] Model and frozen baseline are compared on the same evaluation rows.
- [ ] Validation, recommendations, and honest limitations are complete.
- [ ] The public research paper includes all nine required sections and FlyRank data credit.
- [ ] The deployed paper URL is saved as one line in `submission/paper_url.txt`.

This is an intermediate capstone checkpoint, not the final submission.